### Iterators
Iterators are advanced Python concepts that allow for efficient looping and memory management. Iterators provide a way to access elements of a collection sequentially without exposing the underlying structure.


In [1]:
my_list=[1,2,3,4,5,6]
for i in my_list:
    print(i)

1
2
3
4
5
6


In [2]:
type(my_list)

list

In [3]:
print(my_list)

[1, 2, 3, 4, 5, 6]


In [4]:
## Iterator
iterator=iter(my_list)
print(type(iterator))

<class 'list_iterator'>


In [5]:
iterator

In [6]:
## Iterate through all the element

next(iterator)

1

In [7]:
iterator=iter(my_list)

In [8]:


try:
    print(next(iterator))
except StopIteration:
    print("There are no elements in the iterator")

1


In [9]:
# String iterator
my_string = "Hello"
string_iterator = iter(my_string)

print(next(string_iterator))  # Output: H
print(next(string_iterator))  # Output: e

H
e


---
# 🎯 Iterators: Interview Mastery & Production-Grade Concepts
---
This section covers internal architecture, custom iterator implementations, the `itertools` library, memory profiling, and high-frequency questions asked in Python technical interviews.

### Key Interview Topics:
1. **The Iterator Protocol (`__iter__` and `__next__`)**
2. **Iterable vs. Iterator Distinction (`collections.abc`)**
3. **Custom Iterator Implementations (Countdown & 2D Matrix Flattener)**
4. **Power of `itertools` (`islice`, `chain`, `cycle`, `tee`)**
5. **Memory Profiling: List vs. Iterator (`sys.getsizeof`)**
6. **Why Iterators Cannot Be Rewound (Single-Pass Consumer)**

### 1. The Iterator Protocol Under the Hood
**Interview Question:** *What exact dunder methods must an object implement to be an Iterable vs an Iterator? What happens internally in a `for` loop?*

- **Iterable:** Implements `__iter__()`, which returns an Iterator. (e.g. `list`, `str`, `dict`, `tuple`).
- **Iterator:** Implements:
  1. `__iter__()`: Returns `self` (so an iterator is itself an iterable).
  2. `__next__()`: Returns the next element, or raises `StopIteration` when exhausted.
- **For loop mechanism:**
  ```python
  # for x in obj:
  it = iter(obj) # calls obj.__iter__()
  while True:
      try:
          x = next(it) # calls it.__next__()
      except StopIteration:
          break
  ```

In [10]:
# Demonstrating Iterable vs Iterator
from collections.abc import Iterable, Iterator

numbers = [10, 20, 30]

print("Is list an Iterable?", isinstance(numbers, Iterable))
print("Is list an Iterator?", isinstance(numbers, Iterator))

num_iter = iter(numbers)
print("\nIs iter(numbers) an Iterable?", isinstance(num_iter, Iterable))
print("Is iter(numbers) an Iterator?", isinstance(num_iter, Iterator))

# Calling iter() on an iterator returns itself!
print("iter(num_iter) is num_iter:", iter(num_iter) is num_iter)

Is list an Iterable? True
Is list an Iterator? False

Is iter(numbers) an Iterable? True
Is iter(numbers) an Iterator? True
iter(num_iter) is num_iter: True


### 2. Custom Iterator Implementation
**Interview Question:** *Implement a custom iterator class `Countdown(n)` that counts down from `n` to 1 and raises `StopIteration`.*

In [11]:
# Custom Iterator: Countdown
class Countdown:
    def __init__(self, start):
        self.current = start
        
    def __iter__(self):
        # Must return an iterator object (typically self)
        return self
        
    def __next__(self):
        if self.current <= 0:
            raise StopIteration
        val = self.current
        self.current -= 1
        return val

# Test Countdown
counter = Countdown(4)
print("Manual calls to next():")
print(next(counter))
print(next(counter))

print("\nLooping remaining elements:")
for num in counter:
    print(num)

Manual calls to next():
4
3

Looping remaining elements:
2
1


### 3. Interview Coding Problem: 2D Matrix Flattener Iterator
**Interview Problem:** *Create an iterator `MatrixFlattener` that iterates through elements of a 2D matrix row by row without flattening the entire matrix into a 1D list in memory.*

In [12]:
# 2D Matrix Flattener Iterator (Zero Memory Overhead)
class MatrixFlattener:
    def __init__(self, matrix):
        self.matrix = matrix
        self.row = 0
        self.col = 0
        
    def __iter__(self):
        return self
        
    def __next__(self):
        # Skip any empty rows
        while self.row < len(self.matrix) and self.col >= len(self.matrix[self.row]):
            self.row += 1
            self.col = 0
            
        if self.row >= len(self.matrix):
            raise StopIteration
            
        val = self.matrix[self.row][self.col]
        self.col += 1
        return val

# Test with matrix with varying row lengths
matrix = [
    [1, 2, 3],
    [4],
    [5, 6, 7, 8]
]

flattener = MatrixFlattener(matrix)
print("Flattened elements:")
print(list(flattener))

Flattened elements:
[1, 2, 3, 4, 5, 6, 7, 8]


### 4. Advanced Iteration with `itertools`
**Interview Question:** *How do you slice an iterator, chain multiple streams without copying, or create independent cloned streams?*

- **`itertools.islice(iterable, stop)`:** Slices an iterator lazily without loading the whole sequence.
- **`itertools.chain(*iterables)`:** Lazily chains multiple iterables into one continuous stream.
- **`itertools.tee(iterable, n)`:** Splits a single iterable into $n$ independent iterators!

In [13]:
import itertools

# 1. islice: Extract first 4 elements of an infinite counter
infinite_counter = itertools.count(start=10, step=5)
first_four = list(itertools.islice(infinite_counter, 4))
print("islice from infinite stream:", first_four)

# 2. chain: Combine multiple lists with ZERO copying
a = [1, 2, 3]
b = ['x', 'y', 'z']
chained = list(itertools.chain(a, b))
print("Chained sequence:", chained)

# 3. tee: Clone an iterator into 2 independent consumers
nums = iter([10, 20, 30])
it1, it2 = itertools.tee(nums, 2)

print("Consumer 1:", list(it1))
print("Consumer 2 (still has all elements):", list(it2))

islice from infinite stream: [10, 15, 20, 25]
Chained sequence: [1, 2, 3, 'x', 'y', 'z']
Consumer 1: [10, 20, 30]
Consumer 2 (still has all elements): [10, 20, 30]


### 5. Memory Profiling: List vs Iterator
**Interview Question:** *Compare the memory usage of storing 1,000,000 integers in a list versus generating them via an iterator.*

In [14]:
import sys

# List allocates memory for all 1,000,000 elements upfront:
million_list = list(range(1_000_000))
list_size_mb = sys.getsizeof(million_list) / (1024 ** 2)

# Iterator stores only current state (current number):
million_iter = iter(range(1_000_000))
iter_size_bytes = sys.getsizeof(million_iter)

print(f"List size in RAM:     {list_size_mb:.2f} MB")
print(f"Iterator size in RAM: {iter_size_bytes} bytes")
print(f"Memory Savings:       ~{(list_size_mb * 1024 * 1024 / iter_size_bytes):,.0f}x smaller!")

List size in RAM:     7.63 MB
Iterator size in RAM: 32 bytes
Memory Savings:       ~250,002x smaller!


### 6. Common Interview Trap: Can You Reset an Iterator?
**Interview Question:** *Can you rewind an iterator back to the beginning once exhausted?*

- **NO.** Iterators are strictly single-pass consumers. Once `StopIteration` is raised, calling `next()` will continue to raise `StopIteration`.
- To re-iterate:
  1. Re-call `iter(iterable)` on the original collection.
  2. Or use `itertools.tee()` beforehand if the source cannot be re-instantiated (e.g. streaming socket or file buffer).